Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large 
Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

How to create test datasets

How to run your RAG application on those datasets

How to measure your application's performance using different evaluation metrics
Overview

A typical RAG evaluation workflow consists of three main steps:

Creating a dataset with questions and their expected answers

Running your RAG application on those questions

Using evaluators to measure how well your application performed, looking at factors like:
Answer relevance

Answer accuracy

Retrieval quality

For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

Chatbot Evaluation

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [7]:
LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"

In [9]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbot Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['b379b8e3-a719-4379-ba2c-cf414856145b',
  '9f7f573c-a5ec-4dac-bc60-6c15c9f28633',
  '30c24835-19e9-4246-8634-ec2671e365b6',
  '5cc1c9db-75bb-46b5-91d7-f827d56d0156',
  '24053246-4454-48b8-8e4d-9a8972f0d0f8'],
 'count': 5,
 'as_of': '2026-06-12T05:58:04.499132952Z'}

Define Metrics (LLM As A Judge)

In [10]:
from groq import Groq

# Initialize Groq client
groq_client = Groq()

eval_instructions = (
    "You are an expert professor specialized in grading students' answers to questions."
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
{inputs['question']}

Here is the real answer:
{reference_outputs['answer']}

You are grading the following predicted answer:
{outputs['response']}

Respond with only CORRECT or INCORRECT.

Grade:
"""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # Choose any Groq model
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content}
        ]
    )

    grade = response.choices[0].message.content.strip().upper()

    return grade == "CORRECT"

In [11]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

Run Evaluations

In [12]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model: str = "llama-3.3-70b-versatile", instructions: str = default_instructions) -> str:
    return groq_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [16]:
#call my_app for every datapoints

def ls_target(input:str)->dict:
    return {"response":my_app(input["question"])}

In [17]:
experiment_results=client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="llama-3.3-70b-versatile"
)

View the evaluation results for experiment: 'llama-3.3-70b-versatile-b82cc184' at:
https://smith.langchain.com/o/87d68caf-a391-48ca-a2c6-1fc4f60b9445/datasets/5390b316-a886-4e07-8e22-be8fcee5d30f/compare?selectedSessions=bb1926e7-547a-4a85-88c4-c9f3b9cee39a




5it [00:03,  1.29it/s]
